# 41 -- Section 01e Step 2 rewrite: verify every value before writing it

Reuses notebooks 37/38/39's exact user, movie, seeds and starting weights.
Computes the raw (pre-0.01-scale) random draws for Rumble's weight, and
the weight sum over the other 529 rated movies (excluding Rumble)
directly, as a cross-check against the sum-minus-Rumble shortcut. Does
not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "step2_rewrite_verification" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step2_rewrite_verification.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  40  (40_rumble4_user_mu_list.ipynb)
this notebook is:           41  (41_step2_rewrite_verification.ipynb)


## Part 1 -- Reuse notebook 37/38's exact setup

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
results37_path = out_dir / "37_step1to6_rating_habit_b_results.json"
for p in (train_path, test_path, results31_path, results37_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474

results31 = json.loads(results31_path.read_text())
results37 = json.loads(results37_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
H1_IDX = 0

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
RUMBLE_ID = int(movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)].iloc[0]["movieId"])
rumble_col = movie_to_col[RUMBLE_ID]
MEDIAN_USER = results31["worked_example_plus"]["user"]

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
mask0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
    mask0_full[movie_to_col[row.movieId]] = 1.0
rated_idx = mask0_full.nonzero(as_tuple=True)[0]
a_rumble = v0_full[rumble_col].item()
n_rated = len(rated_idx)
assert n_rated == 530

g_init_p = torch.Generator().manual_seed(REAL_SEED)
raw_plus = torch.randn(n_movies, N_HIDDEN, generator=g_init_p)
W_plus_init = raw_plus * 0.01
g_init_m = torch.Generator().manual_seed(MINUS_SEED)
raw_minus = torch.randn(n_movies, N_HIDDEN, generator=g_init_m)
W_minus_init = raw_minus * 0.01

b_val = results37["step1"]["b"]
raw_plus_rumble = raw_plus[rumble_col, H1_IDX].item()
raw_minus_rumble = raw_minus[rumble_col, H1_IDX].item()
w_plus_rumble = W_plus_init[rumble_col, H1_IDX].item()
w_minus_rumble = W_minus_init[rumble_col, H1_IDX].item()
print(f"a={a_rumble:.6f}  b={b_val:.6f}  n_rated={n_rated}")
print(f"raw_plus_rumble (pre-scale) = {raw_plus_rumble:.6f} -> *0.01 = {w_plus_rumble:.6f}")
print(f"raw_minus_rumble (pre-scale) = {raw_minus_rumble:.6f} -> *0.01 = {w_minus_rumble:.6f}")

a=0.777778  b=0.656813  n_rated=530
raw_plus_rumble (pre-scale) = 0.643322 -> *0.01 = 0.006433
raw_minus_rumble (pre-scale) = -0.936189 -> *0.01 = -0.009362


## Part 2 -- Rumble term, other-529 sum (direct, excluding Rumble), both channels

In [3]:
x_plus_01c = (v0_full @ W_plus_init)[H1_IDX].item()
x_minus_01c = (v0_full @ W_minus_init)[H1_IDX].item()

rumble_term_plus = a_rumble * w_plus_rumble
rumble_term_minus = a_rumble * w_minus_rumble
rest_plus = x_plus_01c - rumble_term_plus
rest_minus = x_minus_01c - rumble_term_minus

other_idx = rated_idx[rated_idx != rumble_col]
assert len(other_idx) == 529
sum_w_plus_other_direct = W_plus_init[other_idx, H1_IDX].sum().item()
sum_w_minus_other_direct = W_minus_init[other_idx, H1_IDX].sum().item()

sum_w_plus_R = W_plus_init[rated_idx, H1_IDX].sum().item()
sum_w_minus_R = W_minus_init[rated_idx, H1_IDX].sum().item()
sum_w_plus_other_shortcut = sum_w_plus_R - w_plus_rumble
sum_w_minus_other_shortcut = sum_w_minus_R - w_minus_rumble

print(f"x+_01c={x_plus_01c:.6f}  Rumble term+={rumble_term_plus:.6f}  rest+={rest_plus:.6f}")
print(f"x-_01c={x_minus_01c:.6f}  Rumble term-={rumble_term_minus:.6f}  rest-={rest_minus:.6f}")
print(f"sum_w_plus_R (530)={sum_w_plus_R:.6f}")
print(f"sum_w_plus_other, direct (529)   = {sum_w_plus_other_direct:.6f}")
print(f"sum_w_plus_other, via subtraction = {sum_w_plus_other_shortcut:.6f}")
print(f"sum_w_minus_R (530)={sum_w_minus_R:.6f}")
print(f"sum_w_minus_other, direct (529)   = {sum_w_minus_other_direct:.6f}")
print(f"sum_w_minus_other, via subtraction = {sum_w_minus_other_shortcut:.6f}")

assert abs(sum_w_plus_other_direct - sum_w_plus_other_shortcut) < 1e-6
assert abs(sum_w_minus_other_direct - sum_w_minus_other_shortcut) < 1e-6
print("\ndirect and subtraction-based other-529 weight sums agree. OK.")

x+_01c=-0.211115  Rumble term+=0.005004  rest+=-0.216119
x-_01c=-0.205367  Rumble term-=-0.007281  rest-=-0.198086
sum_w_plus_R (530)=-0.332340
sum_w_plus_other, direct (529)   = -0.338773
sum_w_plus_other, via subtraction = -0.338773
sum_w_minus_R (530)=-0.325376
sum_w_minus_other, direct (529)   = -0.316015
sum_w_minus_other, via subtraction = -0.316015

direct and subtraction-based other-529 weight sums agree. OK.


## Part 3 -- b term, x+/x- for 01e, cross-checked against notebook 37/38

In [4]:
results38 = json.loads((out_dir / "38_step2_rumble_term_breakdown_results.json").read_text())

b_term_plus = b_val * sum_w_plus_R
b_term_minus = -b_val * sum_w_minus_R
x_plus_01e = x_plus_01c + b_term_plus
x_minus_01e = x_minus_01c + b_term_minus

assert abs(x_plus_01c - results37["step2"]["x_plus"]["01c"]) < 1e-6
assert abs(x_minus_01c - results37["step2"]["x_minus"]["01c"]) < 1e-6
assert abs(x_plus_01e - results37["step2"]["x_plus"]["01e"]) < 1e-6
assert abs(x_minus_01e - results37["step2"]["x_minus"]["01e"]) < 1e-6
assert abs(rumble_term_plus - results38["rumble_term_plus"]) < 1e-9
assert abs(rumble_term_minus - results38["rumble_term_minus"]) < 1e-9
assert abs(rest_plus - results38["rest_plus"]) < 1e-9
assert abs(rest_minus - results38["rest_minus"]) < 1e-9
print(f"b_term_plus (+b*sum_w_plus_R) = {b_term_plus:.6f}")
print(f"b_term_minus (-b*sum_w_minus_R) = {b_term_minus:.6f}")
print(f"x+_01e = {x_plus_01e:.6f}")
print(f"x-_01e = {x_minus_01e:.6f}")
print("\nAll values cross-checked against notebooks 37 and 38. OK.")

b_term_plus (+b*sum_w_plus_R) = -0.218285
b_term_minus (-b*sum_w_minus_R) = 0.213712
x+_01e = -0.429400
x-_01e = 0.008345

All values cross-checked against notebooks 37 and 38. OK.


## Part 4 -- Save for the Step 2 rewrite

In [5]:
results = {
    "notebook": this_name,
    "reused_from": "37_step1to6_rating_habit_b.ipynb, 38_step2_rumble_term_breakdown.ipynb",
    "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0)",
    "a": a_rumble,
    "b": b_val,
    "n_rated": n_rated,
    "n_other": len(other_idx),
    "raw_plus_rumble": raw_plus_rumble,
    "raw_minus_rumble": raw_minus_rumble,
    "w_plus_rumble": w_plus_rumble,
    "w_minus_rumble": w_minus_rumble,
    "rumble_term_plus": rumble_term_plus,
    "rumble_term_minus": rumble_term_minus,
    "rest_plus": rest_plus,
    "rest_minus": rest_minus,
    "x_plus_01c": x_plus_01c,
    "x_minus_01c": x_minus_01c,
    "sum_w_plus_R": sum_w_plus_R,
    "sum_w_minus_R": sum_w_minus_R,
    "sum_w_plus_other": sum_w_plus_other_direct,
    "sum_w_minus_other": sum_w_minus_other_direct,
    "b_term_plus": b_term_plus,
    "b_term_minus": b_term_minus,
    "x_plus_01e": x_plus_01e,
    "x_minus_01e": x_minus_01e,
}
results_path = out_dir / "41_step2_rewrite_verification_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/41_step2_rewrite_verification_results.json
{
  "notebook": "41_step2_rewrite_verification.ipynb",
  "reused_from": "37_step1to6_rating_habit_b.ipynb, 38_step2_rumble_term_breakdown.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0)",
  "a": 0.7777777910232544,
  "b": 0.6568134171907757,
  "n_rated": 530,
  "n_other": 529,
  "raw_plus_rumble": 0.6433217525482178,
  "raw_minus_rumble": -0.9361894130706787,
  "w_plus_rumble": 0.006433217320591211,
  "w_minus_rumble": -0.009361893869936466,
  "rumble_term_plus": 0.005003613556781972,
  "rumble_term_minus": -0.007281473133953331,
  "rest_plus": -0.21611873539821264,
  "rest_minus": -0.1980855406781118,
  "x_plus_01c": -0.21111512184143066,
  "x_minus_01c": -0.20536701381206512,
  "sum_w_plus_R": -0.33233973383903503,
  "sum_w_minus_R": -0.3253764510154724,
  "sum_w_plus_other": -0.3387729525566101,
  "sum_w_mi